In [7]:
import os
import json
from collections import defaultdict, deque
import numpy as np
import pandas as pd
from PIL import Image

# =========================
# PATHS (LOCAL PC)
# =========================
csv_path = r"C:\Users\harol\OneDrive\Desktop\AI_and_ML_and_Quant_Finance\vit_stock_prediction\us_920101-200731_adjusted_full.csv"
base_out = r"C:\Users\harol\OneDrive\Desktop\AI_and_ML_and_Quant_Finance\vit_stock_prediction\image_dataset"
progress_path = r"C:\Users\harol\OneDrive\Desktop\AI_and_ML_and_Quant_Finance\vit_stock_prediction\image_gen_progress.json"

In [8]:
# =========================
# FOLDERS
# =========================
folders = [
    os.path.join(base_out, "train_val", "up"),
    os.path.join(base_out, "train_val", "down"),
    os.path.join(base_out, "test", "up"),
    os.path.join(base_out, "test", "down"),
]

for f in folders:
    os.makedirs(f, exist_ok=True)

In [9]:
# =========================
# SETTINGS
# =========================
lookback = 25
horizon = 20
chunksize = 100_000

train_start = pd.Timestamp("1993-01-01")
train_end = pd.Timestamp("2000-12-31")
test_start = pd.Timestamp("2001-01-01")

price_cols = ["adjusted_open", "adjusted_high", "adjusted_low", "adjusted_close"]

In [10]:
# =========================
# LOAD PROGRESS
# =========================
if os.path.exists(progress_path):
    with open(progress_path, "r") as f:
        progress = json.load(f)
else:
    progress = {
        "last_completed_chunk": 0,
        "saved_count": 0,
        "skipped_count": 0
    }

start_chunk = progress["last_completed_chunk"] + 1
saved_count = progress["saved_count"]
skipped_count = progress["skipped_count"]

print("Resuming from chunk:", start_chunk)
print("Saved so far:", saved_count)
print("Skipped so far:", skipped_count)

Resuming from chunk: 1
Saved so far: 0
Skipped so far: 0


In [11]:
def draw_chart_image(window_df, image_size=224):
    df = window_df.copy()
    df["ma20_close"] = df["adjusted_close"].rolling(window=20, min_periods=1).mean()

    needed = ["adjusted_open", "adjusted_high", "adjusted_low", "adjusted_close", "ma20_close", "VOL"]
    if df[needed].isna().any().any():
        return None

    price_block = df[["adjusted_open", "adjusted_high", "adjusted_low", "adjusted_close", "ma20_close"]]
    pmin = price_block.min().min()
    pmax = price_block.max().max()

    if pd.isna(pmin) or pd.isna(pmax) or pmax <= pmin:
        return None

    img = np.zeros((image_size, image_size, 3), dtype=np.uint8)

    price_top = 0
    price_bottom = 191
    vol_top = 192
    vol_bottom = 223

    n = len(df)
    candle_w = 6
    gap = 2
    step = candle_w + gap
    left_pad = 12

    def scale_price(v):
        return int(price_bottom - (v - pmin) / (pmax - pmin) * (price_bottom - price_top))

    vmax = df["VOL"].max()
    if pd.isna(vmax) or vmax <= 0:
        return None

    def scale_vol(v):
        return int((v / vmax) * (vol_bottom - vol_top))

    ma_points = []
    for i in range(n):
        x = left_pad + i * step + candle_w // 2
        y = scale_price(df.iloc[i]["ma20_close"])
        ma_points.append((x, y))

    for i in range(1, len(ma_points)):
        x1, y1 = ma_points[i - 1]
        x2, y2 = ma_points[i]
        num = max(abs(x2 - x1), abs(y2 - y1)) + 1
        xs = np.linspace(x1, x2, num).astype(int)
        ys = np.linspace(y1, y2, num).astype(int)
        img[ys, xs] = np.array([0, 0, 255], dtype=np.uint8)

    for i in range(n):
        row = df.iloc[i]

        o = row["adjusted_open"]
        h = row["adjusted_high"]
        l = row["adjusted_low"]
        c = row["adjusted_close"]
        v = row["VOL"]

        x_left = left_pad + i * step
        x_center = x_left + candle_w // 2

        yo = scale_price(o)
        yh = scale_price(h)
        yl = scale_price(l)
        yc = scale_price(c)

        top_body = min(yo, yc)
        bot_body = max(yo, yc)

        if c > o:
          color = np.array([0, 255, 0], dtype=np.uint8)   # green
        elif c < o:
          color = np.array([255, 0, 0], dtype=np.uint8)   # red
        else:
          color = np.array([128, 128, 128], dtype=np.uint8)  # grey

        y1, y2 = sorted([yh, yl])
        img[y1:y2 + 1, x_center:x_center + 1] = color

        img[top_body:bot_body + 1, x_left:x_left + candle_w] = color

        vh = scale_vol(v)
        vtop = vol_bottom - vh + 1
        img[vtop:vol_bottom + 1, x_left:x_left + candle_w] = color

    return img

In [ ]:
state = defaultdict(lambda: deque(maxlen=lookback + horizon))

reader = pd.read_csv(csv_path, chunksize=chunksize, low_memory=False)

for chunk_num, chunk in enumerate(reader, start=1):
    if chunk_num < start_chunk:
        continue

    chunk = chunk.copy()
    chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
    chunk["PERMNO"] = pd.to_numeric(chunk["PERMNO"], errors="coerce")
    chunk["VOL"] = pd.to_numeric(chunk["VOL"], errors="coerce")

    for col in price_cols:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    for _, row in chunk.iterrows():
        permno = row["PERMNO"]
        if pd.isna(permno):
            continue

        state[permno].append(row)

        if len(state[permno]) < lookback + horizon:
            continue

        rows = list(state[permno])

        window_rows = rows[:lookback]
        label_row_now = rows[lookback - 1]
        label_row_future = rows[lookback - 1 + horizon]

        window_df = pd.DataFrame(window_rows)

        end_date = label_row_now["date"]
        close_now = label_row_now["adjusted_close"]
        close_future = label_row_future["adjusted_close"]

        if pd.isna(end_date) or pd.isna(close_now) or pd.isna(close_future) or close_now == 0:
            skipped_count += 1
            state[permno].popleft()
            continue

        forward_ret = (close_future - close_now) / close_now
        label = "up" if forward_ret > 0 else "down"

        if train_start <= end_date <= train_end:
            split = "train_val"
        elif end_date >= test_start:
            split = "test"
        else:
            state[permno].popleft()
            continue

        img = draw_chart_image(window_df, image_size=224)
        if img is None:
            skipped_count += 1
            state[permno].popleft()
            continue

        filename = f"{int(permno)}_{pd.Timestamp(end_date).strftime('%Y%m%d')}.png"
        out_path = os.path.join(base_out, split, label, filename)

        if not os.path.exists(out_path):
            Image.fromarray(img).save(out_path)
            saved_count += 1

        state[permno].popleft()

    progress["last_completed_chunk"] = chunk_num
    progress["saved_count"] = saved_count
    progress["skipped_count"] = skipped_count

    with open(progress_path, "w") as f:
        json.dump(progress, f)

    print(f"Finished chunk {chunk_num} | saved: {saved_count} | skipped: {skipped_count}")

print("Done")

Finished chunk 1 | saved: 81349 | skipped: 9457
Finished chunk 2 | saved: 158190 | skipped: 23671
Finished chunk 3 | saved: 240543 | skipped: 31934
Finished chunk 4 | saved: 312338 | skipped: 51229
Finished chunk 5 | saved: 391853 | skipped: 63911
Finished chunk 6 | saved: 461267 | skipped: 84738
Finished chunk 7 | saved: 548549 | skipped: 85212
Finished chunk 8 | saved: 638144 | skipped: 86065
Finished chunk 9 | saved: 718079 | skipped: 95471
Finished chunk 10 | saved: 793810 | skipped: 109410
Finished chunk 11 | saved: 865775 | skipped: 124321
Finished chunk 12 | saved: 946177 | skipped: 132787
Finished chunk 13 | saved: 1034985 | skipped: 136329
Finished chunk 14 | saved: 1102257 | skipped: 157027
Finished chunk 15 | saved: 1174588 | skipped: 170418
Finished chunk 16 | saved: 1242516 | skipped: 193918
Finished chunk 17 | saved: 1314382 | skipped: 209141
Finished chunk 18 | saved: 1396278 | skipped: 216429
Finished chunk 19 | saved: 1477700 | skipped: 228903
Finished chunk 20 | saved